In [ ]:
import re
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_PARAGRAPH_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

def is_title(paragraph):
    text = paragraph.text.strip()
    if not text:
        return False
    if not re.match(r'^[0-9\u0660-\u0669]+', text):
        return False
    if paragraph.alignment != WD_ALIGN_PARAGRAPH.CENTER:
        return False
    word_count = len(text.split())
    if word_count >= 6:
        return False
    return True

def set_rtl(paragraph):
    """ Set the paragraph direction to right-to-left """
    p = paragraph._p  # the underlying lxml element
    pPr = p.get_or_add_pPr()
    bidi = OxmlElement('w:bidi')
    pPr.insert(0, bidi)

def split_and_save_documents(docx_file):
    doc = Document(docx_file)
    current_piece = Document()
    title_text = "Untitled"
    start_new_piece = True

    for paragraph in doc.paragraphs:
        if is_title(paragraph):
            if not start_new_piece:
                # Save the current piece before starting a new one
                filename = re.sub(r'[\\/*?:"<>|]', '_', title_text) + '.docx'
                current_piece.save(filename)
                print(f"Saved: {filename}")
                current_piece = Document()  # Start a new document
            title_text = paragraph.text.strip()
            start_new_piece = False
        new_para = current_piece.add_paragraph(paragraph.text, style=paragraph.style)
        # Ensure RTL layout is preserved
        set_rtl(new_para)

    # Save the last piece of poetry
    if not start_new_piece:
        filename = re.sub(r'[\\/*?:"<>|]', '_', title_text) + '.docx'
        current_piece.save(filename)
        print(f"Saved: {filename}")

if __name__ == "__main__":
    docx_file = 'C:/Users/Sukkar/Desktop/5.docx'  # Replace with your document's filename
    split_and_save_documents(docx_file)
